# <font color='green'><b><u>Single-cell downstream: QC + clustering report<u></b></font>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc

In [ ]:
path_adata = FILE

In [ ]:
adata = sc.read(path_adata)

In [ ]:
from IPython.display import display, HTML

display(HTML("<p>After filtering, the data set contains " + str(adata.n_obs) + " cells and " + str(adata.n_vars) 
             + " genes or transcripts. The number of cells per sample are listed below:</p>"))

In [ ]:
adata.obs.value_counts('sample').rename_axis('samples').reset_index(name='number of cells')

## <font color='green'>Highly variable genes</font>

In [ ]:
sc.pl.highly_variable_genes(adata)

## <font color='green'>PCA</font>

Sample distribution

In [ ]:
sc.pl.pca(
    adata,
    color="sample",
    dimensions=[(0, 1)],
)

QC metrics

In [ ]:
sc.pl.pca(
    adata,
    color=["total_counts", "pct_counts_mt", "pct_counts_ribo"],
    dimensions=[(0, 1)],
)

Loadings indicate the most informative genes across the first PCs

In [ ]:
n_pcs = 6
n_pcs =  adata.obsm['X_pca'].shape[1] if n_pcs > adata.obsm['X_pca'].shape[1] else n_pcs
list_pcs = list(range(1, n_pcs+1))
sc.pl.pca_loadings(adata, include_lowest=False, components=list_pcs)

---
## <font color='green'>UMAP</font>

In [ ]:
adata.obsm['X_pca_umap'] =  adata.obsm['X_umap'].copy()

Samples distribution

In [ ]:
sc.pl.umap(adata, color=['sample'])

In [ ]:
if 'X_umap_scvi' in adata.obsm:
    sc.pl.embedding(adata, basis='X_umap_scvi', color=['sample'], title='scvi UMAP')

QC metrics

In [ ]:
sc.pl.umap(adata, color=['n_genes_by_counts', 'pct_counts_mt'])

In [ ]:
sc.pl.umap(adata, color=['pct_counts_ribo', 'total_counts_hb'])

Distribution of predicted doublets

In [ ]:
sc.pl.umap(adata, color=['predicted_doublet', 'doublet_score'])

---
## <font color='green'>Predicted labels</font>

Celltypist uses pre-trained logistic regression models for predicting labels such as cell types for each cell. Predicted labels of the selected model(s) and the confidence of the predictions are displayed in the UMAPs below. Counts for the predicted labels are summarised in the table(s).

In [ ]:
predicted_labels = [ colname for colname in adata.obs.columns if colname.startswith('celltypist:') ]
for label in predicted_labels:
    sc.pl.umap(adata, color=label)

In [ ]:
from IPython.display import display, HTML
import base64

for label in predicted_labels:
    if not label.endswith(':conf'):
        html = f'<font color="black"><details><summary>Click to expand {label} table</summary>'
        df = adata.obs.value_counts(label).rename_axis(label).reset_index(name='number of cells')
        html += '<div style="overflow-y: scroll; overflow-x: scroll; max-height: 400px">' + df.to_html() + '</div></details>'
        display(HTML(html))

---
## <font color='green'>Clusters</font>

In [ ]:
clustering_label = 'leiden'

# find clusterings
clustering_labels = [ obsname for obsname in adata.obs.columns if obsname.startswith(clustering_label) ]

In [ ]:
# define order of clusters numerically
import numpy as np

list_clusters = np.unique(adata.obs[clustering_label])
list_clusters = sorted([int(x) for x in list_clusters])
list_clusters = [str(x) for x in list_clusters]

In [ ]:
for clustering_label in clustering_labels:
    sc.pl.umap(adata, color=clustering_label, legend_loc='on data')

In [ ]:
if 'X_umap_scvi' in adata.obsm:
    for clustering_label in clustering_labels:
        sc.pl.embedding(adata, basis='X_umap_scvi', color=clustering_label, legend_loc='on data', title=f"scvi UMAP {clustering_label}")